# Cantonese Speech-to-Text Transcription: with Code Samples and Automated Batch Processing Techniques

Here, you'll find the completed full code for the task at hand. However, if you're looking for a more detailed understanding and want to follow along step by step with thorough explanations, we highly recommend you to read our article below:

👉 https://digitalhumanities.hkust.edu.hk/tutorials/transcribe-cantonese-speech-to-text-with-code-samples-and-automated-batch-processing-techniques/

The article provides comprehensive explanations and instructions that will help you grasp the underlying concepts.

## Install and import packages

In [ ]:
!pip install soundfile
!pip install pydub
!pip install SpeechRecognition
!pip install openai --upgrade

In [2]:
import os
import csv
import wave
import soundfile
import requests
import pandas as pd
from datetime import timedelta
from pydub import AudioSegment
import speech_recognition as sr
from transformers import pipeline
from openai import AzureOpenAI

---

## Speech (Cantonese) to Text (Spoken Cantonese)
### Method 1: using Google Speech Recognition

In [3]:
def stt_Cantonese(audio_file):

    # Write the audio data back to the same file, using the original sample rate and a 16-bit PCM encoding to avoid the ValueError: Audio file could not be read as PCM WAV, AIFF/AIFF-C, or Native FLAC; check if file is corrupted or in another format
    data, samplerate = soundfile.read(audio_file)
    soundfile.write(audio_file, data, samplerate, subtype='PCM_16')

    r = sr.Recognizer()

    with sr.AudioFile(audio_file) as source:
        audio = r.record(source)

    try:
        text = r.recognize_google(audio, language="yue-HK") # yue-HK = Cantonese
        return text
    
    except sr.UnknownValueError:
        print("Google Speech Recognition could not understand audio")
        return None
    except sr.RequestError as e:
        print("Could not request results from Google Speech Recognition service; {0}".format(e))
        return None

In [5]:
# Test
print(stt_Cantonese("teashop_1-clip_bossIntro_vocals.wav"))

我姓劉嘅咁我就喺呢條街喺九龍城區出世嘅我58歲呀咁我就做咗喺呢度都係58年以前喺鄉下婆爺呢度就都係做茶囉70年初初就喺對面嘅咁喺對面呢咁以前我哋呢度就全舖後居嘅我哋呢度就瞓七個人後欄嗰度呢咁佢間咗兩間房㗎咁我哋收工呢就喺呢度行啲版呀喺度瞓


### Method 2: Using Whisper Small Cantonese
https://huggingface.co/alvanlii/whisper-small-cantonese

In [6]:
MODEL_NAME = "alvanlii/whisper-small-cantonese" 
lang = "zh"
device = "cpu"  # or "cuda" if you have a GPU
pipe = pipeline(
    task="automatic-speech-recognition",
    model=MODEL_NAME,
    chunk_length_s=30,
    device=device,
)
pipe.model.config.forced_decoder_ids = pipe.tokenizer.get_decoder_prompt_ids(language=lang, task="transcribe")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [8]:
# Test
print(pipe("teashop_1-clip_bossIntro_vocals.wav")["text"])

我姓劉嘅，噉我就喺呢條街喺九城區出世嘅，我五十八歲呀，噉我就做咗喺呢度都係五十八年喇以前喇鄉下太爺都喇到，你都係做茶囉，我哋開開七十年，初初就喺對面嘅，咁喺對面呢通過喺呢度嘅？噉以前我喺呢度就前鋪後居嘅，我哋呢度就瞓七個人，後欄嗰度呢噉佢間咗兩間房嘅，噉我哋收工呢就喺呢度行啲板呀，兩個人相個班喺度瞓覺


## Add punctuation and refine the Spoken Cantonese Text
Using OpenAI

In [9]:
# Parameters
client = AzureOpenAI(
  azure_endpoint = "https://hkust.azure-api.net",
  api_version = "2024-06-01",
  api_key = "<your_azure_openai_api_key>" # See here to see how to get your HKUST Azure OpenAI API key: https://digitalhumanities.hkust.edu.hk/tutorials/how-to-use-hkust-azure-openai-api-key-with-python-with-sample-code-and-use-case-examples/
)

In [13]:
def refineCantonese(message):
    try:
        response = client.chat.completions.create(
            model = 'gpt-4o',
            temperature = 1,
            messages = [
                {"role": "system", "content": """
                                            You are an expert in Cantonese. Please perform the following two steps and return the revised text.
                                            1. Correct potential inaccuracies in the Cantonese text in Hong Kong style. For example, in Hong Kong, text such as '噉', should be written as '咁'.
                                            2. Add punctuation to the text.
                                            """},
                {"role": "user", "content": message}
            ]
        )
        return response.choices[0].message.content
    except:
        return None

In [11]:
# Test - refine the text that generated by Google Speech Recognition
refineCantonese("我姓劉嘅咁我就喺呢條街喺九龍城區出世嘅我58歲呀咁我就做咗喺呢度都係58年以前喺鄉下婆爺呢度就都係做茶囉70年初初就喺對面嘅咁喺對面呢咁以前我哋呢度就全舖後居嘅我哋呢度就瞓七個人後欄嗰度呢咁佢間咗兩間房㗎咁我哋收工呢就喺呢度行啲版呀喺度瞓")

'我姓劉嘅，咁我就喺呢條街，喺九龍城區出世嘅。我58歲呀，咁我就做咗喺呢度，都係58年。以前喺鄉下婆爺呢度，就都係做茶囉。70年初初就喺對面嘅，咁喺對面呢。咁以前我哋呢度，就全舖後居嘅。我哋呢度就瞓七個人，後欄嗰度呢，咁佢間咗兩間房㗎。咁我哋收工呢，就喺呢度行啲版呀，喺度瞓。'

In [14]:
# Test - refine the text that generated by Whisper Small Cantonese
refineCantonese("我姓劉嘅，噉我就喺呢條街喺九城區出世嘅，我五十八歲呀，噉我就做咗喺呢度都係五十八年喇以前喇鄉下太爺都喇到，你都係做茶囉，我哋開開七十年，初初就喺對面嘅，咁喺對面呢通過喺呢度嘅？噉以前我喺呢度就前鋪後居嘅，我哋呢度就瞓七個人，後欄嗰度呢噉佢間咗兩間房嘅，噉我哋收工呢就喺呢度行啲板呀，兩個人相個班喺度瞓覺")

'我姓劉嘅，咁我就喺呢條街，九城區出世嘅。我五十八歲呀，咁我就喺呢度住咗五十八年喇。以前啊，鄉下太爺都嚟過，你都係做茶嘅。我哋開咗七十年，初初就喺對面嘅。咁，對面呢，噉然後通過嚟到呢度㗎。以前我哋就係前鋪後居嘅，我哋呢度瞓七個人。後欄嗰度呢，咁佢間咗兩間房嘅。咁我哋收工呢，就喺呢度行啲板呀，兩個人換番啲喺度瞓覺。'

## Speech (Cantonese) to Text (Written Chinese)
Using Whisper V3: https://huggingface.co/openai/whisper-large-v3

In [17]:
HF_API_URL = "https://api-inference.huggingface.co/models/openai/whisper-large-v3"
headers = {"Authorization": "Bearer <your_hugging_face_api_key>"} #You may get your Hugging Face api key here: https://huggingface.co/settings/tokens

def stt_WrittenChi(audio_file):
    with open(audio_file, "rb") as f:
        data = f.read()
    response = requests.post(HF_API_URL, headers=headers, data=data)
    return response.json()["text"]

In [18]:
# Test
print(stt_WrittenChi("teashop_1-clip_bossIntro_vocals.wav"))

我姓劉,我在九龍城區出生,58歲在這裡工作了58年以前在鄉下,我爺爺也是在這裡工作來到這裡也是做茶,我們開了70年最初在對面,在對面搬過來這裡以前我們這裡是前鋪後居的我們這裡睡7個人後欄那裡,我們建了兩間房子我們下班就在這裡航班板讓我去查詢 讓我回家睡覺


## Text (Spoken Cantonese) to Text (Written Chinese)
Using OpenAI

In [29]:
def Canto_to_Chi_OpenAI(message):
    try:
        response = client.chat.completions.create(
            model = 'gpt-4o',
            temperature = 1,
            messages = [
                {"role": "system", "content": """
                                            Translate the following text from spoken Cantonese text to written language of traditional Chinese text.
                                            """},
                {"role": "user", "content": message}
            ]
        )
        return response.choices[0].message.content
    except:
        return None

In [21]:
# Test
Canto_to_Chi_OpenAI('我姓劉嘅，咁我就喺呢條街、喺九龍城區出世嘅。我58歲啦，咁我就做咗、喺呢度都係58年。以前喺鄉下我阿爺都係做茶，嚟到就都係做茶囉。我哋開咗70年，初初就喺對面嘅，咁喺對面搬過嚟呢度。咁以前我哋呢度就前舖後居嘅，我哋呢度就瞓七個人，後欄嗰度呢咁就間咗兩間房，咁我哋收工呢就喺呢度行啲版呀、茶葉箱呀就喺度瞓。')

'我姓劉，這裡是我在九龍城區出生的。我58歲了，在這裡也幹了58年。以前在鄉下，我爺爺也是做茶的，來到這裡也繼續做茶。我們的店已經開了70年，最初在對面，後來搬到這邊。以前我們這裡是前舖後居，我們七個人住在這裡，後欄那裡隔了兩間房，收工後我們會在這裡鋪木板、茶葉箱，然後在這裡睡覺。'

## Summarize using OpenAI

In [24]:
def summarize(message):
    try:
        response = client.chat.completions.create(
            model = 'gpt-4o',
            temperature = 1,
            messages = [
                {"role": "system", "content": """
                                            You are a journalist. Summarize the following text in traditional Chinese in a concise manner:
                                            """},
                {"role": "user", "content": message}
            ]
        )
        return response.choices[0].message.content
    except:
        return None

In [45]:
# Test
summarize('我姓劉嘅，咁我就喺呢條街、喺九龍城區出世嘅。我58歲啦，咁我就做咗、喺呢度都係58年。以前喺鄉下我阿爺都係做茶，嚟到就都係做茶囉。我哋開咗70年，初初就喺對面嘅，咁喺對面搬過嚟呢度。咁以前我哋呢度就前舖後居嘅，我哋呢度就瞓七個人，後欄嗰度呢咁就間咗兩間房，咁我哋收工呢就喺呢度行啲版呀、茶葉箱呀就喺度瞓。')

'劉先生，58歲，在九龍城區出生並生活至今。他家族從祖父時代開始經營茶葉生意，到現在已有70年歷史。店鋪最初在對面，後來搬到現址。昔日的店鋪是前店後居，劉先生一家七口人住在店鋪後方，用茶葉箱等作臨時床鋪休息。'

---

## Batch processing

In [26]:
# Set the path to the directory containing the WAV files
path = '.' # current folder together with this ipynb file

# Create an empty list to store the data
data = []

# Loop through all files in the directory
for filename in os.listdir(path):
    if filename.endswith('.wav'):
        # Load the WAV file
        audio = AudioSegment.from_wav(os.path.join(path, filename))
        
        # Get the duration of the audio file
        duration = int(audio.duration_seconds)
        duration = str(timedelta(seconds=duration))

        filename_str = str(filename)

        # Speech (Cantonese) to Text (Spoken Cantonese)
        cantonese_text = stt_Cantonese(filename)

        if cantonese_text is not None:
            # Add punctuation and refine the Spoken Cantonese Text
            cantonese_text = refineCantonese(cantonese_text)
        
        # Add the data to the list
        data.append({
            'filename': filename,
            'duration': duration,
            'cantonese_text': cantonese_text,
        })

# DataFrame
df = pd.DataFrame(data)

# Save to a CSV file
df.to_csv('output_CantoneseText.csv', index=False, encoding='utf-8')
print("Exported in a csv file.")

Exported in a csv file.


Then, manually check the Cantonese text and refine. Afterward, perform `Canto_to_Chi_OpenAI` and `summarize` tasks as follows.

In [31]:
# Import the refined Cantonese text
df_refined = pd.read_csv('output_CantoneseText.csv', encoding='utf-8')

# Add two new columns
df_refined['writtenChi_text'] = df_refined['cantonese_text_refined'].apply(Canto_to_Chi_OpenAI)
df_refined['summary'] = df_refined['cantonese_text_refined'].apply(summarize)

# Save to a CSV file
df_refined.to_csv('output_CantoneseText_ChiWrittenText.csv', index=False, encoding='utf-8')
print("Exported in a csv file.")

Exported in a csv file.
